## Step 1: Load and Understanding Dataset

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


In [ ]:
df=sns.load_dataset('tips')
print(f'Rows: {df.shape[0]} Columns: {df.shape[1]}')

#first 5 rows
print(f'First 5 rows:\n{df.head()}')

#cheacking datatype of each column
print(f'Data types:\n{df.dtypes}')

#statistical summary of numerical columns
print(f'Statistical Summary:\n{df.describe()}'  )

#columns info
print(f'Columns Info:\n{df.info()}')

missing_values=df.isnull().sum()
print(f'Missing Values:\n{missing_values}' )

##### the data type for each column is valid and there is no missing values in the dataset

## Step:2  Univariate Analysis

In [ ]:
fig, ax=plt.subplots(2,3,figsize=(15,10 ))

#numerical columns distribution
numerice_col=['total_bill','tip','size']
for i, col in enumerate(numerice_col):
    sns.histplot(data=df, x=col, ax=ax[0,i], color='skyblue', edgecolor='black', kde=True)
    ax[0,i].set_title(f'{col}\n Skewness: {df[col].skew():.2f} Kurtosis: {df[col].kurtosis():.2f} ')

#categorical columns distribution
categoric_col=['sex','smoker','day']
for i, col in enumerate(categoric_col):
    sns.countplot(data=df,x=col,ax=ax[1,i],palette='Set2')
    ax[1,i].set_title(f'{col} Distriution')

plt.suptitle('Distribution  of Tips Dataset', fontsize=16,fontweight='bold')
plt.tight_layout()
plt.show()

#summary of skewness 
for col in numerice_col:
    print(f'{col:15} Skewness: {df[col].skew():.2f}')


##### From the above histogram plots it is concluded that the all numerical columns are right skewed ,which need to be fix before model training depending on the model selection
##### Also the categorical columns are highly imbelanced

## Step:3 Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Numeric vs Numeric 
sns.scatterplot(data=df, x="total_bill", y="tip",
                ax=axes[0, 0], color="steelblue", alpha=0.6)
axes[0, 0].set_title(
    f"Bill vs Tip | corr = {df['total_bill'].corr(df['tip']):.2f}"
)

# Numeric vs Categorical
sns.boxplot(data=df, x="sex", y="total_bill",
            ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Bill by Gender")

sns.boxplot(data=df, x="day", y="total_bill",
            ax=axes[0, 2], palette="Set3")
axes[0, 2].set_title("Bill by Day")

# Categorical vs Categorical
pd.crosstab(df["day"], df["smoker"]).plot(
    kind="bar", ax=axes[1, 0],
    colormap="Set2", stacked=True
)
axes[1, 0].set_title("Day vs Smoker")

sns.violinplot(data=df, x="time", y="tip",
               ax=axes[1, 1], palette="Set1")
axes[1, 1].set_title("Tip by Time")

sns.boxplot(data=df, x="smoker", y="tip",
            ax=axes[1, 2], palette="Set2")
axes[1, 2].set_title("Tip by Smoker")

plt.suptitle("Bivariate Analysis", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

#### Analysis of Bivariate Graphs

1. The scatter plot between total_bill and tip shows a positive correlation (corr = 0.68), which means that as the total bill increases, the tip amount also tends to increase.

2. The box plot of total_bill by gender indicates that male customers generally have slightly higher bills than female customers, although the difference is not very large.

3. The box plot of total_bill by day shows that customers spend more on weekends, especially on Saturday and Sunday, compared to Thursday and Friday.

4. The stacked bar chart of day vs smoker shows that non-smokers are more frequent customers than smokers, and Saturday has the highest number of total customers.

5. The violin plot of tip by time shows that dinner tips are generally higher and more spread out than lunch tips, indicating greater variation in tips during dinner.

6. The box plot of tip by smoker shows that smokers and non-smokers leave similar tips, so smoking status has little effect on tipping behavior.

### Final Conclusion:
Overall, total bill has the strongest influence on tip amount, while factors like gender and smoking status have less impact. Customers tend to spend and tip more during weekends and dinner time.

## Step:4 Multivariate Analysis

In [ ]:
# Corrrelation Heatmap
plt.figure(figsize=(8, 6))
mask=np.triu(np.ones_like(df.select_dtypes(include='number').corr(), dtype=bool))
sns.heatmap(
    df.select_dtypes(include='number').corr(),
    mask=mask,
    annot=True,
    vmax=1.0,
    vmin=-1.0,
    cmap='coolwarm',  
    linewidths=0.5,
    fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


#pairplot
sns.pairplot(df,hue='sex',
             vars=['total_bill','tip','size'],
             palette='Set2',)
plt.suptitle('Pairplot of Tips Dataset', fontsize=16,fontweight='bold')
plt.show()



#### Correlation Heatmap Analysis

1. The heatmap shows the correlation between numeric variables: total_bill, tip, and size.

2. The correlation between total_bill and tip is 0.68, which indicates a strong positive relationship.
   This means that higher bills generally result in higher tips.

3. The correlation between total_bill and size is 0.60, showing that larger group sizes tend to have higher total bills.

4. The correlation between tip and size is 0.49, which is a moderate positive relationship.
   This means that tips increase slightly with group size.

5. All correlations are positive, meaning all variables move in the same direction.

### Final Conclusion:
Total_bill has the strongest relationship with tip, while size also influences both bill and tip but to a lesser extent.

#### Pairplot Analysis (Tips Dataset)

1. The pairplot shows relationships between total_bill, tip, and size, with gender (sex) represented by different colors.

2. The scatter plot between total_bill and tip shows a strong positive relationship, meaning that as the total bill increases, the tip also increases.

3. The relationship between size and total_bill indicates that larger group sizes tend to have higher total bills.

4. The relationship between size and tip shows a slight positive trend, meaning larger groups generally leave higher tips, but the relationship is weaker.

5. The diagonal plots (distribution curves) show that:
   - Most total bills are between 10 and 30.
   - Most tips are between 2 and 4.
   - Most group sizes are between 2 and 4 people.

6. The color distinction (Male vs Female) shows that both genders follow similar patterns, with no major difference in spending or tipping behavior.

### Final Conclusion:
Overall, total_bill is strongly related to tip, and group size also affects both bill and tip. Gender does not significantly impact these relationships.

## Step:5 Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(["total_bill", "tip", "size"]):
    sns.boxplot(data=df, y=col, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Outliers in {col}")
plt.tight_layout()
plt.show()

#### Outlier Analysis

1. The boxplots show the presence of outliers in total_bill, tip, and size.

2. total_bill has several high-value outliers above 40, indicating some customers spent unusually high amounts.

3. tip also contains outliers above 6, showing that some customers gave unusually large tips.

4. size has a few outliers at 5 and 6, indicating rare cases of large groups.

### Conclusion:
Outliers represent extreme values that may affect analysis and should be handled carefully.

## Step: 6 IQR Treatment

In [ ]:
def cap_outliers(df, col):
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df[(df[col] < lower) | (df[col] > upper)])
    df[col] = df[col].clip(lower, upper)
    print(f"{col:15} → {before} outliers capped")
    return df

for col in ["total_bill", "tip"]:
    df = cap_outliers(df, col)

## Step:7 Feature Engineering

In [ ]:
# Tip percentage
df["tip_pct"] = (df["tip"] / df["total_bill"]) * 100

# Bill per person
df["bill_per_person"] = df["total_bill"] / df["size"]

# Is big tipper (tip% > 20%)
df["is_big_tipper"] = (df["tip_pct"] > 20).astype(int)

# Weekend flag
df["is_weekend"] = df["day"].isin(["Sat", "Sun"]).astype(int)

# Dinner flag
df["is_dinner"] = (df["time"] == "Dinner").astype(int)

print(df[["tip_pct", "bill_per_person",
          "is_big_tipper", "is_weekend"]].head())

#### Explanation

1. tip_pct represents the percentage of tip relative to the total bill.
2. bill_per_person shows the average amount spent per person.
3. is_big_tipper is a binary feature indicating whether a customer gives a high tip.
4. is_weekend is a binary feature indicating whether the transaction occurred on a weekend.

### Conclusion:
These features help in better analysis and improve machine learning models by providing more meaningful insights.

## Step:8 Correlation & Statistical Tests

In [ ]:
plt.figure(figsize=(8, 5))

numeric_df = df.select_dtypes(include="number")
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))

sns.heatmap(
    numeric_df.corr(),
    mask=mask, annot=True,
    fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, linewidths=0.5
)
plt.title("Correlation Heatmap — Tips Dataset")
plt.tight_layout()
plt.show()

Total bill & tip are positively correlated → higher bill, higher tip.
Group size increases total bill and tip, but reduces tip %.
Tip % decreases as total bill increases.
Dinner & weekend are strongly related → higher spending time.
Bill per person decreases with larger group size.

In [ ]:
# ── T-Test: Does gender affect total bill? ──
male   = df[df["sex"] == "Male"]["total_bill"]
female = df[df["sex"] == "Female"]["total_bill"]
t, p   = stats.ttest_ind(male, female)
print(f"Gender vs Bill        → p = {p:.4f} "
      f"{'✅ Significant' if p < 0.05 else '❌ Not significant'}")

# ── T-Test: Does smoking affect tip? ────────
smoker    = df[df["smoker"] == "Yes"]["tip"]
nonsmoker = df[df["smoker"] == "No"]["tip"]
t, p      = stats.ttest_ind(smoker, nonsmoker)
print(f"Smoker vs Tip         → p = {p:.4f} "
      f"{'✅ Significant' if p < 0.05 else '❌ Not significant'}")

# ── T-Test: Does time affect bill? ──────────
dinner = df[df["time"] == "Dinner"]["total_bill"]
lunch  = df[df["time"] == "Lunch"]["total_bill"]
t, p   = stats.ttest_ind(dinner, lunch)
print(f"Time vs Bill          → p = {p:.4f} "
      f"{'✅ Significant' if p < 0.05 else '❌ Not significant'}")

# ── Chi-Square: Day vs Smoker ────────────────
crosstab        = pd.crosstab(df["day"], df["smoker"])
chi2, p, dof, _ = stats.chi2_contingency(crosstab)
print(f"Day vs Smoker         → p = {p:.4f} "
      f"{'✅ Significant' if p < 0.05 else '❌ Not significant'}")

# ── Chi-Square: Sex vs Smoker ────────────────
crosstab        = pd.crosstab(df["sex"], df["smoker"])
chi2, p, dof, _ = stats.chi2_contingency(crosstab)
print(f"Sex vs Smoker         → p = {p:.4f} "
      f"{'✅ Significant' if p < 0.05 else '❌ Not significant'}")

# ── ANOVA: Bill across Days ──────────────────
groups  = [df[df["day"] == d]["total_bill"] for d in df["day"].unique()]
f, p    = stats.f_oneway(*groups)
print(f"Day vs Bill (ANOVA)   → p = {p:.4f} "
      f"{'✅ Significant' if p < 0.05 else '❌ Not significant'}")

Gender vs Bill → Significant (p = 0.0244)
Time vs Bill → Significant (p = 0.0038)
Day vs Smoker → Significant (p = 0.0000)
Day vs Bill → Significant (p = 0.0387)
Smoker vs Tip → Not significant
Sex vs Smoker → Not significant

## TIPS DATASET — FINAL EDA INSIGHTS 
1. total_bill & tip strongly correlated (0.68)   
    → Higher bill = higher tip                    
                                                  
2. Gender does NOT affect bill (p > 0.05)     
                                                  
3. Smoking does NOT affect tip (p > 0.05)     
                                                  
4. Dinner bills HIGHER than Lunch (p < 0.05)  
                                                  
5. Day does NOT significantly affect bill     
                                                  
6. total_bill & tip are RIGHT SKEWED             
    → Need log transform before linear modeling  

### Handling The skewness

In [ ]:
print("Skewness BEFORE transformation:")
for col in ["total_bill", "tip", "size"]:
    print(f"  {col:15} skew = {df[col].skew():.2f}")

In [ ]:
# total_bill → sqrt (moderate skew)
df["total_bill"] = np.sqrt(df["total_bill"])

# tip → sqrt (moderate skew)
df["tip"] = np.sqrt(df["tip"])

# size → log1p (high skew > 1)
df["size"] = np.log1p(df["size"])

# Check after
print("\nAFTER:")
for col in ["total_bill", "tip", "size"]:
    print(f"  {col:15} skew = {df[col].skew():.2f}")

Since size is still at 0.94 after log1p, try Box-Cox which automatically finds the best transformation:

In [ ]:
df["size"], fitted_lambda = stats.boxcox(df["size"])

print(f"Best lambda found: {fitted_lambda:.4f}")
print(f"size skew AFTER Box-Cox: {df['size'].skew():.2f}")

size skew AFTER Box-Cox = -0.05  , Almost perfectly normal!

### Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

le = LabelEncoder()
df_encoded["sex"]    = le.fit_transform(df["sex"])
df_encoded["smoker"] = le.fit_transform(df["smoker"])
df_encoded["time"]   = le.fit_transform(df["time"])

df_encoded = pd.get_dummies(df_encoded, columns=["day"], drop_first=True)

bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(" Encoding done! Ready for Scaling.")
print(df_encoded.head())

### Feature Scaling

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
X = df_encoded.drop("tip", axis=1)
y = df_encoded["tip"]

print("Ranges BEFORE scaling:")
print(X[["total_bill", "size"]].describe().round(2))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

#### Apply RobustScaler

In [ ]:
cols_to_scale = ["total_bill", "size"]

scaler = RobustScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])
#fit_transform on train, transform on test to avoid data leakage

print("\nRanges AFTER scaling:")
print(X_train[cols_to_scale].describe().round(2))

### Feature Selection

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature"   : X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.round(3))

#### Drop All Leaked Features

In [ ]:
# Drop leakage columns
leakage_cols = ["tip_pct", "is_big_tipper"]

X_train_final = X_train.drop(columns=leakage_cols, errors="ignore")
X_test_final  = X_test.drop(columns=leakage_cols, errors="ignore")

print("Dropped leakage columns:", leakage_cols)
print(f"New shape: {X_train_final.shape}")

### Re-run Feature Importance Without Leakage

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_final, y_train)

importance = pd.DataFrame({
    "Feature"   : X_train_final.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.round(3))

#### Apply Threshold — Keep importance > 0.03

In [ ]:


# Visualize with threshold line
plt.figure(figsize=(10, 5))
sns.barplot(data=importance, x="Importance", y="Feature", palette="viridis")
plt.axvline(x=0.03, color="red", linestyle="--", label="Threshold (0.03)")
plt.title("Feature Importance — After Leakage Fix")
plt.legend()
plt.tight_layout()
plt.show()

# Keep features above threshold
threshold = 0.03
selected = importance[importance["Importance"] >= threshold]["Feature"].tolist()
print(f"Selected features: {selected}")

In [ ]:
# Apply selection
X_train_final = X_train_final[selected]
X_test_final  = X_test_final[selected]

print(f"Features selected: {selected}")
print(f"X_train shape: {X_train_final.shape}")
print(f"X_test shape:  {X_test_final.shape}")

#### The dataset is now clean, lean, and ready for modeling with 4 meaningful features.